## 부록 : 멀티에이전트와 시스템 고도화

03 데이터 조회 에이전트와 04 RAG 에이전트를 대상으로 아래 내용을 다룹니다.

1. 멀티에이전트 개념과 대표 구조
2. 03·04 에이전트를 슈퍼바이저 구조로 묶기
3. 기존 시스템의 고도화 방향
4. 고도화 적용 : RAG 질문 재작성, 데이터 조회 도구의 스키마·예시 쿼리

### 0. 준비 : 03·04 에이전트 다시 만들기

03, 04 노트북에서 만든 에이전트를 이 노트북 안에서 다시 만듭니다.

* 데이터 : `./db/fab_etch.db` (03과 같은 DB), `./04_RAG_agent/docs` (04에서 전처리를 마친 문서)

In [ ]:
import os
import sqlite3

import pandas as pd
from dotenv import load_dotenv, find_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

# .env 파일로부터 api_key를 불러오는 코드
load_dotenv(find_dotenv())

model = ChatOpenAI(
    model="gpt-4o-mini",
    base_url=os.getenv("gpt_4o_mini_BASE_URL"),
    temperature=0,
)

**03 데이터 조회 에이전트** : 03 노트북에서 마지막으로 만든 `run_sql`과 같습니다.

In [ ]:
DB_PATH = "./db/fab_etch.db"


@tool
def run_sql(query: str) -> str:
    """ETCH 공정 DB(SQLite)에 SQL 쿼리를 실행하고 결과를 돌려줍니다.

    테이블: etch_lot (한 행 = Lot 1건의 ETCH 공정 결과)
    컬럼:
    - equipment_id  : 설비 ID (ETCH_01, ETCH_02, ETCH_03)
    - chamber_id    : 설비 내 챔버 ID (CH_A, CH_B, CH_C)
    - temperature_c : 공정 온도 기록값 (°C)
    """
    conn = sqlite3.connect(DB_PATH)
    try:
        df = pd.read_sql(query, conn)
    except Exception as e:
        return f"쿼리 실행 오류: {e}"
    finally:
        conn.close()
    return df.to_string(index=False)


DATA_PROMPT = "너는 반도체 공정 데이터를 분석하는 어시스턴트다. 데이터에 대한 질문은 run_sql 도구로 DB를 조회해서 답해라."

data_agent = create_agent(model, tools=[run_sql], system_prompt=DATA_PROMPT)

**04 RAG 에이전트** : 보도자료(md)는 제목 구조로, 8대 공정 문서(txt)는 문서 하나를 청크 하나로 나누고, BM25와 Dense를 반씩 섞은 Hybrid 검색기를 사용합니다.

In [ ]:
from pathlib import Path

from kiwipiepy import Kiwi
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter

DOCS_DIR = Path("./04_RAG_agent/docs")


def load_chunks():
    """md는 제목(#, ##, ###) 단위로, txt는 파일 하나를 청크 하나로 나누고 앞에 파일명을 붙임"""
    chunks = []
    md_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[("#", "제목"), ("##", "첨부"), ("###", "소제목")], strip_headers=False)
    for path in sorted(DOCS_DIR.glob("*.md")):
        for part in md_splitter.split_text(path.read_text(encoding="utf-8")):
            chunks.append(Document(f"[{path.name}]\n{part.page_content}", metadata={"source": path.name}))
    for path in sorted(DOCS_DIR.glob("*.txt")):
        chunks.append(Document(f"[{path.name}]\n{path.read_text(encoding='utf-8')}", metadata={"source": path.name}))
    for i, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = i
    return chunks


kiwi = Kiwi()
kiwi.add_user_word("수율", "NNG")


def tokenize(text):
    return [t.form for t in kiwi.tokenize(text) if t.tag.startswith(("NN", "SL", "SN"))]


chunks = load_chunks()

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small", base_url=os.getenv("Embedding_BASE_URL"))
bm25_retriever = BM25Retriever.from_documents(chunks, preprocess_func=tokenize, k=4)
dense_retriever = FAISS.from_documents(chunks, embedding_model).as_retriever(search_kwargs={"k": 4})
hybrid_retriever = EnsembleRetriever(retrievers=[bm25_retriever, dense_retriever], weights=[0.5, 0.5])

print(f"청크 수: {len(chunks)}개")

In [ ]:
def format_docs(docs):
    return "\n\n---\n\n".join(f"[청크 {d.metadata['chunk_id']}]\n{d.page_content}" for d in docs)


@tool
def search_documents(query: str) -> str:
    """반도체 8대 공정 설명 자료와 반도체 첨단패키징 보도자료에서 질문과 관련된 문서를 검색합니다.

    query: 검색할 내용을 담은 짧은 문장이나 키워드 (예: "EDS 공정 불량 칩 판정")
    """
    return format_docs(hybrid_retriever.invoke(query)[:4])


RAG_PROMPT = """너는 반도체 공정과 산업 정책 자료를 안내하는 사내 어시스턴트다.
- 질문에 답하기 전에 반드시 search_documents 도구로 관련 문서를 검색해라.
- 검색된 문서에 있는 내용만으로 답하고, 문서에 없으면 "자료에서 찾을 수 없습니다"라고 답해라.
"""

rag_agent = create_agent(model, tools=[search_documents], system_prompt=RAG_PROMPT)

에이전트가 호출한 도구와 입력값을 출력하는 함수입니다. 이 노트북 전체에서 사용합니다.

In [ ]:
def show_steps(response):
    """도구 호출(이름, 입력값)과 최종 답변을 순서대로 출력"""
    for message in response["messages"]:
        for call in getattr(message, "tool_calls", None) or []:
            print(f"▶ {call['name']} : {list(call['args'].values())[0]}")
    print("\n답변 :\n" + response["messages"][-1].content)

### 1. 멀티에이전트

**멀티에이전트(Multi-Agent)** : 역할을 나눈 여러 에이전트가 함께 하나의 작업을 처리하는 구조입니다.

에이전트 하나에 도구와 규칙을 계속 추가하면 아래 문제가 생깁니다.

| 문제 | 내용 |
|---|---|
| 도구 선택 오류 | 도구가 많아질수록 LLM이 상황에 맞지 않는 도구를 고르는 경우가 늘어남 |
| 규칙 누락 | 시스템 프롬프트에 여러 업무의 규칙이 섞이면 일부 규칙을 지키지 않음 |
| 긴 대화 기록 | SQL 결과표, 검색된 문서 원문이 대화 기록에 계속 쌓여 비용이 늘고 앞의 내용을 놓침 |

역할별로 에이전트를 나누면 각 에이전트는 자기 역할에 필요한 도구와 규칙만 가집니다.

**대표 구조**

| 구조 | 동작 방식 | 순서를 정하는 주체 |
|---|---|---|
| 슈퍼바이저 (Supervisor) | 관리 에이전트가 전문 에이전트를 도구처럼 호출하고 결과를 모아 답변 | LLM (관리 에이전트) |
| 라우터 (Router) | 질문을 먼저 분류하고, 분류 결과에 해당하는 에이전트에게 전달 | 분류 결과 + 코드 |
| 핸드오프 (Handoff) | 에이전트가 대화를 다른 에이전트에게 넘기고, 넘겨받은 에이전트가 이어서 사용자와 대화 | LLM (각 에이전트) |
| 워크플로 (Workflow) | 에이전트 실행 순서를 코드(LangGraph 그래프)로 고정 | 사람 |

**슈퍼바이저 구조**

```
                      ┌─ ask_data_agent(요청문) → 데이터 조회 에이전트 (run_sql)
[질문] → 슈퍼바이저 ──┤
                      └─ ask_rag_agent(요청문)  → RAG 에이전트 (search_documents)
       → 두 결과를 모아 답변
```

### 2. 03·04 에이전트를 슈퍼바이저로 묶기

에이전트를 실행하고 최종 답변만 돌려주는 함수를 `@tool`로 감싸면, 슈퍼바이저는 이 함수를 일반 도구와 같은 방식으로 호출합니다. 기존 `data_agent`, `rag_agent` 코드는 수정하지 않습니다.

* 도구 설명(docstring) : 슈퍼바이저가 어떤 에이전트를 부를지 판단하는 기준
* `request` : 슈퍼바이저가 전문 에이전트에게 보내는 요청문. 전문 에이전트는 이 요청문만 보고 작업함

In [ ]:
@tool
def ask_data_agent(request: str) -> str:
    """ETCH 공정 DB(2026년 8월 Lot 데이터: 설비, 챔버, 제품, 레시피, 압력, 온도, 불량률 등)를 조회하는 담당에게 일을 맡긴다.
    수치 집계나 비교가 필요할 때 사용한다. request에는 조회할 내용과 조건을 구체적으로 적는다."""
    result = data_agent.invoke({"messages": [{"role": "user", "content": request}]})
    return result["messages"][-1].content


@tool
def ask_rag_agent(request: str) -> str:
    """반도체 8대 공정 설명 자료와 첨단패키징 보도자료를 검색하는 담당에게 일을 맡긴다.
    공정 원리, 용어, 정책·사업 내용이 필요할 때 사용한다. request에는 찾을 내용을 구체적으로 적는다."""
    result = rag_agent.invoke({"messages": [{"role": "user", "content": request}]})
    return result["messages"][-1].content


supervisor = create_agent(
    model,
    tools=[ask_data_agent, ask_rag_agent],
    system_prompt="""너는 반도체 Fab 엔지니어의 질문에 답하는 관리 담당이다. 데이터와 문서를 직접 보지 않고 담당에게 일을 맡긴다.
- 공정 데이터 수치가 필요하면 ask_data_agent, 공정 설명이나 정책 자료가 필요하면 ask_rag_agent를 호출한다.
- 둘 다 필요하면 둘 다 호출한다.
- 담당이 돌려준 내용만 사용해서 답한다.""",
)

질문 종류에 따라 슈퍼바이저가 호출하는 에이전트를 확인합니다.

In [ ]:
# 데이터만 필요한 질문
response = supervisor.invoke({"messages": [{"role": "user", "content": "8월에 불량률이 가장 높은 설비와 챔버는?"}]})
show_steps(response)

In [ ]:
# 문서만 필요한 질문
response = supervisor.invoke({"messages": [{"role": "user", "content": "식각 공정에서 웨이퍼 부위별로 불량이 생기는 이유는?"}]})
show_steps(response)

In [ ]:
# 둘 다 필요한 질문
question = "8월에 불량률이 가장 높은 설비와 챔버를 찾고, 식각 공정에서 불량이 생기는 원인을 자료에서 찾아서 같이 정리해줘"

response = supervisor.invoke({"messages": [{"role": "user", "content": question}]})
show_steps(response)

슈퍼바이저의 대화 기록에는 SQL 결과표나 문서 원문 대신 전문 에이전트가 정리한 답변만 남습니다.

In [ ]:
for message in response["messages"]:
    print(f"[{message.type}] 글자 수 {len(str(message.content))}")

**정리**

| 항목 | 내용 |
|---|---|
| 연결 방법 | 기존 에이전트를 `@tool` 함수로 감싸서 슈퍼바이저의 도구로 등록 |
| 장점 | 역할별 도구·규칙 분리, 슈퍼바이저의 대화 기록이 짧게 유지됨, 에이전트를 추가할 때 도구 하나만 추가 |
| 주의점 | LLM 호출 횟수 증가(시간·비용), 요청문에 조건이 빠지면 전문 에이전트가 다른 내용을 찾음, 호출 순서와 횟수가 질문마다 달라짐 |

### 3. 기존 시스템의 고도화 방향

**RAG 시스템**

| 방향 | 내용 |
|---|---|
| 질문 재작성 (Multi-Query) | 사용자 질문을 LLM이 검색어 여러 개로 바꿔서 검색하고 결과를 합침. 복합 질문, 표현이 문서와 다른 질문에 사용 |
| 재순위화 (Rerank) | 검색된 청크를 별도 모델(Cross-Encoder 등)로 질문과의 관련도를 다시 계산해서 순서를 조정 |
| 메타데이터 필터 | 청크에 문서 종류, 날짜, 부서 등을 붙이고 검색할 때 조건으로 범위를 좁힘 |
| 출처 인용 강제 | 답변의 각 문장에 근거 청크 번호나 파일명을 표시하게 하고, 인용이 없으면 다시 작성 |
| 검색 평가셋 | 질문과 정답 청크를 짝지은 목록을 만들어 청킹·검색 방식을 바꿀 때마다 정답 청크 포함 비율을 측정 |

**데이터 조회 시스템**

| 방향 | 내용 |
|---|---|
| 스키마·예시 쿼리 제공 | 모든 컬럼의 의미·단위·값 형식, 계산 규칙, 예시 쿼리를 도구 설명에 포함 |
| 정해진 쿼리 도구 | 자주 쓰는 분석은 쿼리를 미리 작성해 두고 에이전트는 기간·설비 같은 값만 입력 |
| 안전장치 | 읽기 전용 연결, SELECT만 허용, 반환 행 수 제한 |
| 결과 검증 | 조회 결과가 비어 있거나 값의 범위가 이상하면 쿼리를 다시 확인하게 함 |

**공통**

| 방향 | 내용 |
|---|---|
| 미들웨어 | 도구 호출 순서 강제, 호출 횟수 제한, 개인정보 마스킹 |
| 사람 승인 | 되돌리기 어려운 작업은 실행 전 사람이 승인 (`HumanInTheLoopMiddleware`) |
| 로그·추적 | 질문, 도구 호출, 답변을 기록해서 실패 사례를 모으고 수정에 사용 (LangSmith 등) |

아래에서는 **질문 재작성**과 **스키마·예시 쿼리 제공**을 적용합니다.

### 4-1. RAG 고도화 : 질문 재작성 (Multi-Query)

한 질문에 여러 내용이 섞여 있으면, 질문 전체를 검색어 하나로 사용했을 때 일부 내용에 해당하는 청크만 검색됩니다.

**적용 전** : 질문을 그대로 검색어로 사용

In [ ]:
question = "정부가 첨단패키징에 지원하는 금액은 얼마고, 패키징 공정은 어떤 일을 하는 공정이야?"


def show_sources(docs):
    for d in docs:
        preview = d.page_content.split("\n", 1)[1][:50].replace("\n", " ")
        print(f"청크 {d.metadata['chunk_id']:>2} | {d.metadata['source'][:30]:<30} | {preview}")


show_sources(hybrid_retriever.invoke(question)[:4])

**적용 후** : LLM이 질문을 검색어 여러 개로 나눈 뒤, 검색어마다 검색한 결과를 합칩니다.

1. 구조화된 출력(`with_structured_output`)으로 검색어 목록 생성
2. 검색어마다 `hybrid_retriever`로 검색
3. 같은 청크는 한 번만 남기고 합침

In [ ]:
from pydantic import BaseModel, Field


class SearchQueries(BaseModel):
    queries: list[str] = Field(
        description="문서 검색에 사용할 검색어 목록(1~3개). 질문에 여러 내용이 있으면 내용마다 하나씩 만든다. "
                    "각 검색어는 핵심 명사 위주의 짧은 문장으로 쓴다."
    )


query_writer = model.with_structured_output(SearchQueries)


def multi_query_search(question, k=3):
    queries = query_writer.invoke(
        f"다음 질문에 답하는 데 필요한 문서를 찾기 위한 검색어를 만들어줘.\n질문: {question}"
    ).queries
    print("생성된 검색어 :", queries)

    results, seen = [], set()
    for q in queries:
        for doc in hybrid_retriever.invoke(q)[:k]:
            if doc.metadata["chunk_id"] not in seen:
                seen.add(doc.metadata["chunk_id"])
                results.append(doc)
    return results


show_sources(multi_query_search(question))

검색 도구에 적용하고 RAG 에이전트를 다시 만들어서 답변을 비교합니다.

In [ ]:
@tool
def search_documents_mq(query: str) -> str:
    """반도체 8대 공정 설명 자료와 반도체 첨단패키징 보도자료에서 질문과 관련된 문서를 검색합니다.

    query: 사용자의 질문 (여러 내용이 섞여 있어도 그대로 넣는다)
    """
    return format_docs(multi_query_search(query))


rag_agent_mq = create_agent(model, tools=[search_documents_mq], system_prompt=RAG_PROMPT)

print("===== 적용 전 =====")
show_steps(rag_agent.invoke({"messages": [{"role": "user", "content": question}]}))
print("\n===== 적용 후 =====")
show_steps(rag_agent_mq.invoke({"messages": [{"role": "user", "content": question}]}))

| 항목 | 내용 |
|---|---|
| 효과 | 복합 질문의 각 내용에 해당하는 청크를 모두 검색 |
| 비용 | 검색할 때마다 검색어 생성용 LLM 호출 1회 추가, 에이전트에게 넘기는 청크 수 증가 |

### 4-2. 데이터 조회 고도화 : 스키마·예시 쿼리 제공

03의 `run_sql` 설명에는 컬럼 3개만 적혀 있습니다. 설명에 없는 컬럼이 필요한 질문을 하면, 에이전트는 컬럼 이름을 추측하거나 테이블 구조를 먼저 조회합니다.

**적용 전**

In [ ]:
question = "레시피별 평균 압력과 불량률을 알려줘"


def show_sql_steps(response):
    """실행한 쿼리, 오류 여부, 최종 답변을 출력"""
    calls = 0
    for message in response["messages"]:
        if message.type == "ai":
            for call in message.tool_calls:
                calls += 1
                print(f"[쿼리 {calls}]\n{call['args']['query']}")
        elif message.type == "tool" and message.content.startswith("쿼리 실행 오류"):
            print(f"  → {message.content[:120]}")
    print(f"\nrun_sql 호출 횟수 : {calls}")
    print("\n답변 :\n" + response["messages"][-1].content)


show_sql_steps(data_agent.invoke({"messages": [{"role": "user", "content": question}]}))

**적용 후** : 도구 설명에 아래 내용을 추가합니다.

* 모든 컬럼의 의미, 단위, 값 형식 (`db/fab_etch_data_dictionary.md` 기준)
* 계산 규칙 : 불량률 집계 방법, SQLite 정수 나눗셈
* 예시 쿼리 : 자주 쓰는 집계 형태

함께 안전장치(읽기 전용 연결, 반환 행 수 제한)도 적용합니다.

In [ ]:
MAX_ROWS = 100


@tool
def run_sql_v2(query: str) -> str:
    """ETCH 공정 DB(SQLite)에 SELECT 쿼리를 실행하고 결과를 돌려줍니다.

    테이블: etch_lot (한 행 = 웨이퍼 50장 Lot 1건의 ETCH 공정 결과, 2026-08-01 ~ 2026-08-31, 11,160행)

    컬럼:
    - timestamp      (TEXT)    : 공정 완료 시각, 'YYYY-MM-DD HH:MM:SS'
    - lot_id         (TEXT)    : Lot 식별자
    - product_id     (TEXT)    : 제품군 코드 (PROD_A12, PROD_B07, PROD_C03, PROD_D09)
    - equipment_id   (TEXT)    : 설비 ID (ETCH_01, ETCH_02, ETCH_03)
    - chamber_id     (TEXT)    : 설비 내 챔버 ID (CH_A, CH_B, CH_C)
    - recipe_id      (TEXT)    : 공정 레시피 ID (RCP_A12, RCP_B07, RCP_C03, RCP_D09)
    - wafer_count    (INTEGER) : Lot 내 처리 웨이퍼 수(개)
    - pressure_kpa   (REAL)    : 압력(kPa)
    - temperature_c  (REAL)    : 공정 온도(°C)
    - rf_power_w     (REAL)    : RF Power(W)
    - gas_flow_sccm  (REAL)    : 가스 유량(sccm)
    - process_time_s (REAL)    : 공정 시간(초)
    - defect_count   (INTEGER) : 불량 웨이퍼 수(개)
    - defect_rate    (REAL)    : defect_count / wafer_count, 비율(0~1)
    - yield_rate     (REAL)    : 1 - defect_rate, 비율(0~1)

    계산 규칙:
    - 불량률 집계는 SUM(defect_count) * 1.0 / SUM(wafer_count)로 계산한다.
    - SQLite는 정수끼리 나누면 소수점을 버리므로 * 1.0을 붙인다.
    - 날짜 조건은 date(timestamp)로 비교한다.

    예시 쿼리:
    - 챔버별 불량률(%):
      SELECT equipment_id, chamber_id, ROUND(SUM(defect_count) * 100.0 / SUM(wafer_count), 2) AS 불량률_pct
      FROM etch_lot GROUP BY equipment_id, chamber_id ORDER BY 불량률_pct DESC
    - 특정 설비의 날짜별 평균 압력:
      SELECT date(timestamp) AS 날짜, ROUND(AVG(pressure_kpa), 2) AS 평균압력
      FROM etch_lot WHERE equipment_id = 'ETCH_03' GROUP BY 날짜 ORDER BY 날짜

    조회만 가능하며, 결과는 최대 100행까지 돌려준다.
    """
    conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)  # 읽기 전용 연결
    try:
        df = pd.read_sql(query, conn)
    except Exception as e:
        return f"쿼리 실행 오류: {e}"
    finally:
        conn.close()
    if len(df) > MAX_ROWS:
        return f"(전체 {len(df)}행 중 앞 {MAX_ROWS}행만 표시)\n" + df.head(MAX_ROWS).to_string(index=False)
    return df.to_string(index=False)


data_agent_v2 = create_agent(model, tools=[run_sql_v2], system_prompt=DATA_PROMPT)

show_sql_steps(data_agent_v2.invoke({"messages": [{"role": "user", "content": question}]}))

읽기 전용 연결에서는 데이터를 바꾸는 쿼리가 실행되지 않습니다.

In [ ]:
print(run_sql_v2.invoke({"query": "DELETE FROM etch_lot WHERE equipment_id = 'ETCH_03'"}))

| 항목 | 내용 |
|---|---|
| 효과 | 컬럼 이름 추측·테이블 구조 조회 단계가 줄어듦, 집계 방식이 일정해짐 |
| 비용 | 도구 설명이 길어져 매 호출마다 입력 토큰 증가 |
| 한계 | 테이블·컬럼이 많은 DB는 설명 전체를 넣기 어려움 → 질문과 관련된 테이블 설명만 검색해서 넣는 방식 사용 |

두 고도화 모두 에이전트 구조는 그대로 두고 도구 내부만 바꿨습니다. 앞의 슈퍼바이저에서 `data_agent`, `rag_agent`를 `data_agent_v2`, `rag_agent_mq`로 바꾸면 멀티에이전트 구조에도 같은 개선이 반영됩니다.